# 00_Data ingest and sanity checks
Converts the June–September 2025 Citi Bike trip files to Parquet, profiles their size and schema, cleans trips, defines the study area and its station capacities, and checks the net-flow pattern that motivates overnight rebalancing.

## 1. Load data

Converts each month's raw CSV files to one Parquet file in `data/processed/`. The conversion is skipped automatically if the Parquet file already exists. Add a month to `months` to load more data.

| Name | Source | Role |
|---|---|---|
| `trips_raw` | `data/raw/<yyyymm>/*.csv` (Citi Bike System Data) | One row per trip |
| `stations` | `data/external/station_information_20261101.json` (GBFS) | Station capacity and location, snapshot taken 2026-11-01 |

In [35]:
import os
import duckdb

duckdb.sql("SET enable_progress_bar = false")
duckdb.sql("SET memory_limit = '4GB'")  # 8GB machine; leave room for the OS
raw_dir = '../data/raw'
external_dir = '../data/external'
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)

months = ['202506', '202507', '202508', '202509']

# Skip if the Parquet file already exists; convert only on the first run
for month in months:
    parquet_path = f'{processed_dir}/trips_{month}.parquet'
    if os.path.exists(parquet_path):
        print(f'{month}: Parquet already exists, skipping conversion.')
    else:
        duckdb.sql(f"""
        COPY (
            SELECT *
            FROM read_csv('{raw_dir}/{month}/*.csv',
                          union_by_name = true,
                          types = {{'start_station_id': 'VARCHAR',
                                   'end_station_id':   'VARCHAR'}})
        )
        TO '{parquet_path}' (FORMAT PARQUET)
        """)
        print(f'{month}: conversion done.')

202506: conversion done.
202507: Parquet already exists, skipping conversion.
202508: conversion done.
202509: conversion done.


### Register the `trips_raw` and `stations` views
`trips_raw` reads every monthly Parquet file. `stations` flattens the GBFS JSON to one row per station; trip data uses the GBFS `short_name` (e.g. `6140.05`) as its station ID, so `short_name` becomes `station_id`. Re-run this cell after a kernel restart.

In [36]:
duckdb.sql(f"""
CREATE OR REPLACE VIEW trips_raw AS
SELECT * FROM '{processed_dir}/trips_*.parquet';

CREATE OR REPLACE VIEW stations AS
SELECT s.short_name AS station_id,
       s.name,
       s.lat,
       s.lon AS lng,
       s.capacity
FROM (
    SELECT unnest(data.stations) AS s
    FROM read_json('{external_dir}/station_information_20261101.json')
);
""")

## 2. Data profiling

### Schema check
Station IDs are read as `VARCHAR` so values like `6140.05` keep their format.

In [37]:
duckdb.sql("DESCRIBE SELECT * FROM trips_raw").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ ride_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rideable_type      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ started_at         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ ended_at           │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_id   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_name   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_lat          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │

### Row count and date range

In [38]:
duckdb.sql("""
SELECT COUNT(*)        AS n_trips,
       MIN(started_at) AS first_start,
       MAX(started_at) AS last_start
FROM trips_raw
""").show()

┌──────────┬─────────────────────────┬─────────────────────────┐
│ n_trips  │       first_start       │       last_start        │
│  int64   │        timestamp        │        timestamp        │
├──────────┼─────────────────────────┼─────────────────────────┤
│ 20156378 │ 2025-05-30 23:47:29.703 │ 2025-09-30 23:58:26.908 │
└──────────┴─────────────────────────┴─────────────────────────┘



### Missing stations
Trips with no start or end station, mostly e-bikes not returned to a dock.

In [39]:
duckdb.sql("""
SELECT COUNT(*) FILTER (WHERE start_station_id IS NULL) AS null_start,
       COUNT(*) FILTER (WHERE end_station_id IS NULL)   AS null_end,
       ROUND(100.0 * COUNT(*) FILTER (WHERE start_station_id IS NULL) / COUNT(*), 2) AS null_start_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE end_station_id IS NULL)   / COUNT(*), 2) AS null_end_pct
FROM trips_raw
""").show()

┌────────────┬──────────┬────────────────┬──────────────┐
│ null_start │ null_end │ null_start_pct │ null_end_pct │
│   int64    │  int64   │     double     │    double    │
├────────────┼──────────┼────────────────┼──────────────┤
│       9219 │    63147 │           0.05 │         0.31 │
└────────────┴──────────┴────────────────┴──────────────┘



### Bike type and rider type
`SUM(COUNT(*)) OVER ()` gives the overall total as the denominator.

In [40]:
duckdb.sql("""
SELECT rideable_type,
       member_casual,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM trips_raw
GROUP BY rideable_type, member_casual
ORDER BY n DESC
""").show()

┌───────────────┬───────────────┬──────────┬────────┐
│ rideable_type │ member_casual │    n     │  pct   │
│    varchar    │    varchar    │  int64   │ double │
├───────────────┼───────────────┼──────────┼────────┤
│ electric_bike │ member        │ 11098808 │   55.1 │
│ classic_bike  │ member        │  4894369 │   24.3 │
│ electric_bike │ casual        │  3097093 │   15.4 │
│ classic_bike  │ casual        │  1066108 │    5.3 │
└───────────────┴───────────────┴──────────┴────────┘



### Trip duration
1st, 50th and 99th percentiles in minutes, used to set the cleaning thresholds.

In [41]:
duckdb.sql("""
SELECT quantile_cont(date_diff('minute', started_at, ended_at), [0.01, 0.5, 0.99]) AS duration_min_q
FROM trips_raw
""").show()

┌───────────────────┐
│  duration_min_q   │
│     double[]      │
├───────────────────┤
│ [1.0, 10.0, 64.0] │
└───────────────────┘



### Station count

In [42]:
duckdb.sql("""
SELECT COUNT(DISTINCT start_station_id) AS n_stations
FROM trips_raw
""").show()

┌────────────┐
│ n_stations │
│   int64    │
├────────────┤
│       2184 │
└────────────┘



**Key findings: data profiling**
- The June–September 2025 files hold 20,156,378 trips. The earliest trip starts on 2025-05-30, so files are grouped by end time and a few trips cross months.
- Missing station IDs are rare: 9,219 start (0.05%) and 63,147 end (0.31%).
- E-bikes account for 70.5% of trips and members for 79.4%.
- The median trip lasts 10 minutes; the 1st and 99th percentiles are 1 and 64 minutes.
- Trips start at 2,184 stations across the system.

## 3. Cleaning

### Register the `trips_clean` view
Keeps trips that start within the study period and last 1 minute to 3 hours. Trips under 1 minute are usually re-docks; trips over 3 hours are usually unreturned bikes. Trips with a missing end station are kept, because their departure is still real demand at the start station; arrival counts exclude them later.

In [43]:
start_date, end_date = '2025-06-01', '2025-10-01'

duckdb.sql(f"""
CREATE OR REPLACE VIEW trips_clean AS
SELECT *,
       date_diff('second', started_at, ended_at) / 60.0 AS duration_min
FROM trips_raw
WHERE started_at >= '{start_date}'
  AND started_at <  '{end_date}'
  AND date_diff('second', started_at, ended_at) BETWEEN 60 AND 3 * 3600
""")

### Rows removed

In [44]:
duckdb.sql("""
WITH counts AS (
    SELECT (SELECT COUNT(*) FROM trips_raw)   AS n_raw,
           (SELECT COUNT(*) FROM trips_clean) AS n_clean
)
SELECT n_raw,
       n_clean,
       ROUND(100.0 * (1 - n_clean / n_raw), 2) AS removed_pct
FROM counts
""").show()

┌──────────┬──────────┬─────────────┐
│  n_raw   │ n_clean  │ removed_pct │
│  int64   │  int64   │   double    │
├──────────┼──────────┼─────────────┤
│ 20156378 │ 20124050 │        0.16 │
└──────────┴──────────┴─────────────┘



### GBFS match rate
Share of cleaned trips whose start station appears in the GBFS snapshot. The snapshot is from November 2026, so stations removed or renumbered since mid-2025 do not match.

In [45]:
duckdb.sql("""
SELECT ROUND(100.0 * COUNT(s.station_id) / COUNT(*), 1) AS match_pct
FROM trips_clean t
LEFT JOIN stations s
       ON t.start_station_id = s.station_id
""").show()

┌───────────┐
│ match_pct │
│  double   │
├───────────┤
│      96.4 │
└───────────┘



**Key findings: cleaning**
- Cleaning keeps 20,124,050 trips and removes 0.16% (cross-period starts, trips under 1 minute or over 3 hours).
- 96.4% of cleaned trips start at a station in the GBFS snapshot; the remaining stations need a capacity fallback.

## 4. Study area

### Station locations
One row per start station. Location is the median start coordinate, so stations no longer in GBFS are still placed.

In [46]:
duckdb.sql("""
CREATE OR REPLACE TABLE station_loc AS
SELECT start_station_id              AS station_id,
       any_value(start_station_name) AS name,
       median(start_lat)             AS lat,
       median(start_lng)             AS lng,
       COUNT(*)                      AS n_departures
FROM trips_clean
WHERE start_station_id IS NOT NULL
GROUP BY start_station_id
""")

### Candidate areas
Midtown alone is mostly offices. Extending south to 14th St adds residential areas (Chelsea, Murray Hill, Kips Bay), so the area contains both morning outflow and inflow stations.

In [47]:
duckdb.sql("""
WITH boxes (area, lat_min, lat_max, lng_min, lng_max) AS (
    VALUES ('Midtown (34th-59th St)', 40.745, 40.770, -74.010, -73.965),
           ('14th-59th St',           40.730, 40.770, -74.012, -73.965)
)
SELECT b.area,
       COUNT(*)            AS n_stations,
       SUM(l.n_departures) AS n_departures
FROM boxes b
JOIN station_loc l
  ON l.lat BETWEEN b.lat_min AND b.lat_max
 AND l.lng BETWEEN b.lng_min AND b.lng_max
GROUP BY b.area
ORDER BY n_stations
""").show()

┌────────────────────────┬────────────┬──────────────┐
│          area          │ n_stations │ n_departures │
│        varchar         │   int64    │    int128    │
├────────────────────────┼────────────┼──────────────┤
│ Midtown (34th-59th St) │        136 │      3495805 │
│ 14th-59th St           │        229 │      6167451 │
└────────────────────────┴────────────┴──────────────┘



### Register `study_stations` with capacity
Keeps stations in the 14th–59th St box. Capacity comes from the GBFS ID match first, then a name match for renumbered stations, then the area median. `capacity_source` records which rule was used.

In [48]:
duckdb.sql("""
CREATE OR REPLACE TABLE study_stations AS
WITH area AS (
    SELECT *
    FROM station_loc
    WHERE lat BETWEEN 40.730 AND 40.770
      AND lng BETWEEN -74.012 AND -73.965
),
by_name AS (
    -- capacity looked up by station name, for stations whose ID changed
    SELECT name, any_value(capacity) AS cap_by_name
    FROM stations
    GROUP BY name
),
joined AS (
    SELECT a.*,
           s.capacity    AS cap_by_id,
           n.cap_by_name
    FROM area a
    LEFT JOIN stations s ON a.station_id = s.station_id
    LEFT JOIN by_name n  ON a.name = n.name
)
SELECT station_id,
       name,
       lat,
       lng,
       n_departures,
       COALESCE(cap_by_id, cap_by_name,
                (SELECT median(cap_by_id) FROM joined))::INT AS capacity,
       CASE WHEN cap_by_id   IS NOT NULL THEN 'gbfs_id'
            WHEN cap_by_name IS NOT NULL THEN 'gbfs_name'
            ELSE 'imputed_median'
       END AS capacity_source
FROM joined
""")

### Capacity source

In [49]:
duckdb.sql("""
SELECT capacity_source,
       COUNT(*)          AS n_stations,
       SUM(n_departures) AS n_departures,
       ROUND(100.0 * SUM(n_departures) / SUM(SUM(n_departures)) OVER (), 1) AS departures_pct
FROM study_stations
GROUP BY capacity_source
ORDER BY n_stations DESC
""").show()

┌─────────────────┬────────────┬──────────────┬────────────────┐
│ capacity_source │ n_stations │ n_departures │ departures_pct │
│     varchar     │   int64    │    int128    │     double     │
├─────────────────┼────────────┼──────────────┼────────────────┤
│ gbfs_id         │        211 │      5827006 │           94.5 │
│ imputed_median  │         14 │       322349 │            5.2 │
│ gbfs_name       │          4 │        18096 │            0.3 │
└─────────────────┴────────────┴──────────────┴────────────────┘



**Key findings: study area**
- Midtown alone has 136 stations. The 14th–59th St area has 229 stations and 6,167,451 departures, and is used from here on.
- Capacity: 211 stations matched by GBFS ID, 4 by name, 14 imputed with the area median. Imputed stations carry 322,349 departures (5.2%), so the optimization needs a capacity sensitivity check.

## 5. Net flow patterns

### Weekday count
Number of weekdays in the study period, used to turn totals into daily averages. Juneteenth, July 4 and Labor Day are counted as weekdays here; holidays are handled as a feature in forecasting.

In [50]:
n_weekdays = duckdb.sql("""
SELECT COUNT(DISTINCT started_at::DATE)
FROM trips_clean
WHERE dayofweek(started_at) BETWEEN 1 AND 5
""").fetchall()[0][0]
n_weekdays

87

### Morning peak station types
Average weekday net flow (arrivals − departures) per station from 7 to 10 AM. Stations below −5 bikes per day are outflow stations; stations above +5 are inflow stations.

In [51]:
duckdb.sql(f"""
WITH am AS (
    -- departures count as -1, arrivals as +1
    SELECT start_station_id AS station_id, -1 AS flow
    FROM trips_clean
    WHERE hour(started_at) BETWEEN 7 AND 9
      AND dayofweek(started_at) BETWEEN 1 AND 5
    UNION ALL
    SELECT end_station_id, 1
    FROM trips_clean
    WHERE hour(ended_at) BETWEEN 7 AND 9
      AND dayofweek(ended_at) BETWEEN 1 AND 5
      AND end_station_id IS NOT NULL
),
net AS (
    SELECT station_id,
           SUM(flow) / {n_weekdays} AS avg_daily_net
    FROM am
    WHERE station_id IN (SELECT station_id FROM study_stations)
    GROUP BY station_id
)
SELECT COUNT(*) FILTER (WHERE avg_daily_net < -5)          AS outflow_stations,
       COUNT(*) FILTER (WHERE avg_daily_net > 5)           AS inflow_stations,
       COUNT(*) FILTER (WHERE avg_daily_net BETWEEN -5 AND 5) AS balanced_stations,
       ROUND(SUM(avg_daily_net), 1)                        AS area_total_net
FROM net
""").show()

┌──────────────────┬─────────────────┬───────────────────┬────────────────┐
│ outflow_stations │ inflow_stations │ balanced_stations │ area_total_net │
│      int64       │      int64      │       int64       │     double     │
├──────────────────┼─────────────────┼───────────────────┼────────────────┤
│               73 │             118 │                38 │         2159.4 │
└──────────────────┴─────────────────┴───────────────────┴────────────────┘



### Daily net flow cycle
Average weekday net flow for the whole area by hour. `cum_net` is the running total from midnight.

In [52]:
duckdb.sql(f"""
WITH flows AS (
    SELECT hour(started_at) AS hr, -1 AS flow
    FROM trips_clean
    WHERE dayofweek(started_at) BETWEEN 1 AND 5
      AND start_station_id IN (SELECT station_id FROM study_stations)
    UNION ALL
    SELECT hour(ended_at), 1
    FROM trips_clean
    WHERE dayofweek(ended_at) BETWEEN 1 AND 5
      AND end_station_id IN (SELECT station_id FROM study_stations)
)
SELECT hr,
       ROUND(SUM(flow) / {n_weekdays}, 1)                        AS avg_net,
       ROUND(SUM(SUM(flow)) OVER (ORDER BY hr) / {n_weekdays}, 1) AS cum_net
FROM flows
GROUP BY hr
ORDER BY hr
""").show(max_rows=24)

┌───────┬─────────┬─────────┐
│  hr   │ avg_net │ cum_net │
│ int64 │ double  │ double  │
├───────┼─────────┼─────────┤
│     0 │   -34.3 │   -34.3 │
│     1 │   -16.1 │   -50.4 │
│     2 │   -11.2 │   -61.6 │
│     3 │     1.7 │   -59.9 │
│     4 │    -2.7 │   -62.6 │
│     5 │   -20.5 │   -83.1 │
│     6 │    33.4 │   -49.7 │
│     7 │   322.6 │   272.9 │
│     8 │   903.0 │  1175.9 │
│     9 │   933.9 │  2109.8 │
│    10 │   255.4 │  2365.1 │
│    11 │    33.6 │  2398.8 │
│    12 │   -14.2 │  2384.5 │
│    13 │   -77.9 │  2306.6 │
│    14 │  -141.0 │  2165.6 │
│    15 │  -224.3 │  1941.3 │
│    16 │  -638.5 │  1302.8 │
│    17 │ -1177.4 │   125.4 │
│    18 │  -574.2 │  -448.8 │
│    19 │  -207.8 │  -656.6 │
│    20 │  -111.7 │  -768.3 │
│    21 │  -130.5 │  -898.9 │
│    22 │  -101.4 │ -1000.3 │
│    23 │   -74.1 │ -1074.4 │
└───────┴─────────┴─────────┘
  24 rows         3 columns



**Key findings: net flow patterns**
- The study period has 87 weekdays.
- Weekday mornings (7–10 AM): 73 outflow, 118 inflow and 38 balanced stations, with a net gain of 2,159.4 bikes for the area. Rebalancing within the area is possible, but the binding morning constraint is free docks at inflow stations.
- Daily cycle: net inflow peaks at +2,399 bikes by noon, then reverses, with the largest hourly outflow at 5 PM (−1,177 bikes).
- The area ends each weekday at −1,074 bikes. Trip data excludes the operator's own truck moves, so the operator must resupply this area; the optimization model needs an external depot node. Weekend flows are not yet checked and may offset part of this loss.
- From midnight to 6 AM hourly net flow stays within ±35 bikes, a natural overnight rebalancing window.

## 6. Save outputs

### Save `study_stations`
Writes the station list to Parquet for the forecasting and optimization notebooks.

In [53]:
duckdb.sql(f"""
COPY study_stations TO '{processed_dir}/study_stations.parquet' (FORMAT PARQUET)
""")